In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
df = pd.read_csv("../Dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()
df.shape
df.info()
df.describe()
df.columns
df.isnull().sum()

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isnull().sum()
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)
df.drop("customerID", axis=1, inplace=True)


In [ ]:
df.replace("No internet service", "No", inplace=True)
df.replace("No phone service", "No", inplace=True)
df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.info()

In [ ]:
sns.countplot(x="Churn", data=df)

plt.title("Customer Churn Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x="Contract",
    hue="Churn",
    data=df
)

plt.title("Contract Type vs Churn")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x="tenure",
    hue="Churn",
    kde=True
)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    x="InternetService",
    hue="Churn",
    data=df
)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.countplot(
    y="PaymentMethod",
    hue="Churn",
    data=df
)

plt.show()

In [ ]:
df["Churn"] = df["Churn"].map({
    "Yes":1,
    "No":0
})

In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    df[
        ["tenure","MonthlyCharges","TotalCharges","Churn"]
    ].corr(),
    annot=True,
    cmap="coolwarm"
)

plt.show()

In [ ]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0,12,24,48,100],
    labels=[
        "0-12",
        "13-24",
        "25-48",
        "49+"
    ]
)

In [ ]:
df["avg_monthly_spend"] = (
    df["TotalCharges"] /
    df["tenure"].replace(0,1)
)

In [ ]:
df = pd.get_dummies(
    df,
    drop_first=True
)

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [ ]:
df["avg_monthly_spend"] = np.where(
    df["tenure"] == 0,
    0,
    df["TotalCharges"] / df["tenure"]
)

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy_score(
    y_test,
    y_pred
)

In [ ]:
precision_score(
    y_test,
    y_pred
)

In [ ]:
recall_score(
    y_test,
    y_pred
)

In [ ]:
f1_score(
    y_test,
    y_pred
)

In [ ]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:
coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

coef.sort_values(
    by="Coefficient",
    ascending=False,
    inplace=True
)

coef.head(15)

In [ ]:
import pickle

with open("model.pkl", "wb") as file:
    pickle.dump(model, file)

with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [ ]:
print(X.columns.tolist())